# 03 압축·보정·평가

SurvFace: TPIR@20/FPIR. TinyFace·SCFace: rank. LFW·RFW: TAR/FMR. 같은 fit/safety의 Global-safe와 Continuous를 비교합니다. 최종 test에서 threshold를 선택하지 않습니다. KEEP_RAW_RESULTS=False는 요약 검증 후 이번 실행의 상세 결과만 정리합니다.


In [ ]:
# 1. 사용자 설정
# 목적: 기준 실험/보정 표본 수 분석 단계와 데이터셋·출력 위치·상세 결과 보존 여부를 정합니다.

from pathlib import Path
import sys, os
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research").is_dir())  # 현재 ronbun 위치를 자동 탐색하는 내부 경로입니다. 직접 바꾸지 않습니다.
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
CONFIG_PATH = PROJECT_ROOT / "configs/experiments/facex_workflow.yaml"  # 실험 YAML 경로입니다. 다른 실험 설정을 쓰려면 같은 workflow 형식의 YAML을 지정합니다.
# FACE_DATASET_ROOT/FACE_MODEL_ROOT 환경변수가 우선합니다. 미지정 시 Windows F:/, Linux /data/ 기본값을 읽습니다.
from research.runtime.storage import roots
STORAGE = roots()  # 환경변수/OS 기본값에서 읽은 경로입니다. 실제 경로 변경은 아래 두 ROOT 변수에서 합니다.
DATASET_ROOT = Path(STORAGE["DATASET_ROOT"])  # 공유 데이터 루트; 필요하면 Path("실제 데이터 경로")로 변경합니다. Windows 기본 F:/face_datasets.
MODEL_ROOT = Path(STORAGE["MODEL_ROOT"])  # 공유 모델 루트; 필요하면 Path("실제 모델 경로")로 변경합니다. Windows 기본 F:/models.
os.environ["FACE_DATASET_ROOT"] = str(DATASET_ROOT)
os.environ["FACE_MODEL_ROOT"] = str(MODEL_ROOT)
DATASETS = ["survface", "tinyface", "scface", "lfw", "rfw"]  # 이 다섯 이름 중 평가할 대상을 중복 없이 선택합니다(일부만 선택 가능).
EXECUTE = False  # False=상태/계획 확인, True=아래 셀에서 실제 작업 실행입니다.
STUDY_STAGE = "baseline"  # "baseline"=기준 실험, "calibration_size"=기준 실험 완료 후 동일 모델·codec의 보정 표본 수 분석.
BASELINE_REPORT = None  # baseline은 None; calibration_size는 완료된 기준 실험의 report_dir를 Path 또는 문자열로 지정합니다.
# calibration_size에서는 DATASETS를 survface/lfw/rfw 중에서 선택합니다. TinyFace/SCFace는 baseline의 rank 평가 대상입니다.
# 원본 데이터·모델·02 입력 cache·이전 완료 결과는 이 보존 옵션의 정리 대상이 아닙니다.
KEEP_RAW_RESULTS = True  # True=평가 상세 결과 보존, False=요약 검증 후 이번 실행의 상세 결과 정리입니다.
OUTPUT_ROOT = DATASET_ROOT.parent / "ronbun_results/facex_workflow"  # 입력 cache와 실험 결과의 공통 저장 루트입니다. 02·03에서 같은 경로를 사용합니다.


In [ ]:
# 2. 실험 실행 및 결과 경로 확인
# 목적: 선택 단계의 실험을 실행하거나 계획을 확인하고, 실행 시 정확한 report_dir를 받습니다.

from research.experiments.facex_workflow import run_workflow, inspect_workflow
if EXECUTE:
    result = run_workflow(CONFIG_PATH, datasets=DATASETS, output_root=OUTPUT_ROOT,
                          keep_raw_results=KEEP_RAW_RESULTS, study_stage=STUDY_STAGE,
                          baseline_report=BASELINE_REPORT)
    REPORT_DIR = result["report_dir"]  # 실행 결과가 자동으로 정하는 경로입니다. 04의 REPORT_DIR에 그대로 지정합니다.
    display(result)
else:
    print("Study stage:", STUDY_STAGE, "Baseline:", BASELINE_REPORT)
    display(inspect_workflow(CONFIG_PATH, datasets=DATASETS))


기준 실험 완료 후 동일 모델·입력·codec을 고정하여 calibration_size를 실행합니다. 표본 수 분석은 원래 fit 인물 내부에서 fit/safety/validation을 분리하고 최종 safety와 test는 사용하지 않습니다. 데이터가 부족한 조건도 제외하지 않고 insufficient_rows로 기록합니다. 반복 fold/subset은 독립 표본이 아닙니다. 결과는 FPIR/FMR 안정성 진단이며 최적 FR 배분이나 TPIR/TAR 개선 증거가 아닙니다.
